# Bluestock Fintech — Advanced Analytics + Risk Metrics
## Capstone Project I: Mutual Fund Analytics
**Author:** Surendhar | **Cohort:** 2025 | **Due:** 02 Jul 2026


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")

nav = pd.read_csv("data/nav_history.csv", parse_dates=["date"])
perf = pd.read_csv("data/scheme_performance.csv")
txn = pd.read_csv("data/investor_transactions.csv", parse_dates=["transaction_date"])
holdings = pd.read_csv("data/portfolio_holdings.csv")
nav = nav.sort_values(["amfi_code","date"]).reset_index(drop=True)
nav = nav.merge(perf[["amfi_code","scheme_name","risk_grade","sharpe_ratio","category"]], on="amfi_code", how="left")
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
nav = nav.dropna(subset=["daily_return"])
print(f"Data loaded! NAV: {nav.shape} | Txn: {txn.shape}")

## Task 1 — Historical VaR (95%) and CVaR

In [ ]:
var_results = []
for code, group in nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    var_95 = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()
    var_results.append({
        "amfi_code": code,
        "scheme_name": group["scheme_name"].iloc[0],
        "var_95_pct": round(var_95 * 100, 4),
        "cvar_95_pct": round(cvar_95 * 100, 4),
        "risk_grade": group["risk_grade"].iloc[0]
    })
var_df = pd.DataFrame(var_results).sort_values("var_95_pct")
var_df.to_csv("var_cvar_report.csv", index=False)
print("VaR and CVaR computed for all 40 funds")
print(var_df[["scheme_name","var_95_pct","cvar_95_pct","risk_grade"]].to_string(index=False))

## Task 2 — Rolling 90-Day Sharpe Ratio

In [ ]:
from IPython.display import Image
Image("charts/rolling_sharpe_chart.png", width=900)

## Task 3 — Investor Cohort Analysis

In [ ]:
sip = txn[txn["transaction_type"]=="SIP"].copy()
txn["first_year"] = txn.groupby("investor_id")["transaction_date"].transform("min").dt.year
cohort = txn.groupby(["investor_id","first_year"]).agg(
    total_invested=("amount_inr","sum"), sip_count=("transaction_type","count")).reset_index()
sip_cohort = sip.groupby("investor_id").agg(avg_sip=("amount_inr","mean")).reset_index()
cohort = cohort.merge(sip_cohort, on="investor_id", how="left")
cohort_summary = cohort.groupby("first_year").agg(
    total_investors=("investor_id","count"),
    avg_sip_amount=("avg_sip","mean"),
    avg_total_invested=("total_invested","mean")).reset_index()
print(cohort_summary.to_string(index=False))

## Task 4 — SIP Continuity Analysis

In [ ]:
sip_sorted = sip.sort_values(["investor_id","transaction_date"])
sip_sorted["prev_date"] = sip_sorted.groupby("investor_id")["transaction_date"].shift(1)
sip_sorted["gap_days"] = (sip_sorted["transaction_date"] - sip_sorted["prev_date"]).dt.days
sip_count = sip.groupby("investor_id").size().reset_index(name="sip_count")
frequent_sip = sip_count[sip_count["sip_count"] >= 6]["investor_id"].tolist()
sip_gaps = sip_sorted[sip_sorted["investor_id"].isin(frequent_sip)].groupby("investor_id")["gap_days"].mean().reset_index()
sip_gaps.columns = ["investor_id","avg_gap_days"]
sip_gaps["at_risk"] = sip_gaps["avg_gap_days"] > 35
at_risk_count = sip_gaps["at_risk"].sum()
total_frequent = len(sip_gaps)
continuity_rate = round((1 - at_risk_count/total_frequent)*100, 2)
print(f"Total frequent SIP investors: {total_frequent}")
print(f"At-risk investors (gap > 35 days): {at_risk_count}")
print(f"SIP Continuity Rate: {continuity_rate}%")
print(sip_gaps.head(10).to_string(index=False))

## Task 5 — Simple Fund Recommender

In [ ]:
def recommend_funds(risk_appetite):
    risk_map = {"Low": ["Low"], "Moderate": ["Moderate"], "High": ["High","Very High"]}
    if risk_appetite not in risk_map:
        return "Invalid. Enter Low, Moderate, or High."
    filtered = perf[perf["risk_grade"].isin(risk_map[risk_appetite])]
    top3 = filtered.nlargest(3,"sharpe_ratio")[["scheme_name","fund_house","category","sharpe_ratio","risk_grade","expense_ratio_pct"]].reset_index(drop=True)
    top3.index += 1
    return top3

print("LOW risk recommendations:")
print(recommend_funds("Low").to_string())
print("
MODERATE risk recommendations:")
print(recommend_funds("Moderate").to_string())
print("
HIGH risk recommendations:")
print(recommend_funds("High").to_string())

## Task 6 — Sector HHI Concentration

In [ ]:
holdings["weight_sq"] = (holdings["weight_pct"]/100)**2
hhi_df = holdings.groupby("amfi_code")["weight_sq"].sum().reset_index()
hhi_df.columns = ["amfi_code","hhi_score"]
hhi_df = hhi_df.merge(perf[["amfi_code","scheme_name","category"]], on="amfi_code", how="left")
hhi_df["concentration_level"] = hhi_df["hhi_score"].apply(lambda x: "High" if x > 0.15 else ("Moderate" if x > 0.08 else "Low"))
hhi_df = hhi_df.sort_values("hhi_score", ascending=False)
print(hhi_df[["scheme_name","hhi_score","concentration_level"]].to_string(index=False))

## Task 7 — 5 Advanced Insights

1. **Small Cap Funds Have Highest VaR**: SBI Small Cap Fund has the highest VaR at -2.69%, meaning on 5% of trading days, losses exceed this threshold. Small Cap funds consistently appear in the top 5 highest risk funds, confirming their volatile nature. [Ref: Task 1]

2. **Liquid Funds Dominate Risk-Adjusted Returns**: ICICI Pru Liquid Fund leads with a Sharpe ratio of 7.68, far above all other categories. This suggests liquid funds offer the best return per unit of risk, making them ideal for Low risk investors. [Ref: Task 5]

3. **2024 Cohort Drives the Most Investment**: The majority of investors (4,803) made their first transaction in 2024, with an average total investment of ₹7.27L. The 2025 cohort shows a higher avg SIP amount (₹13,652) suggesting newer investors are investing larger amounts. [Ref: Task 3]

4. **SIP Continuity is a Major Concern**: Out of 1,362 frequent SIP investors, 1,332 (97.8%) are flagged as at-risk with average gaps exceeding 35 days. This indicates most investors are not maintaining consistent monthly SIP schedules, which could significantly impact long-term wealth creation. [Ref: Task 4]

5. **Axis Bluechip and ABSL Small Cap Show Highest Portfolio Concentration**: With HHI scores above 0.20, these funds have highly concentrated sector exposure. High concentration means higher risk if any particular sector underperforms. Investors should consider funds with lower HHI scores for better diversification. [Ref: Task 6]
